01. Setup

In [1]:
# Instalação do Java e Spark
!apt-get update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark

# Variáveis de ambiente
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

# Pacote Iceberg
!wget -q https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.6.1/iceberg-spark-runtime-3.5_2.12-1.6.1.jar -P /content/spark-3.5.0-bin-hadoop3/jars/

import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col

# Sessão Spark
spark = SparkSession.builder \
.appName("IcebergWithSpark") \
.config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
.config("spark.sql.catalog.hadoop_catalog", "org.apache.iceberg.spark.SparkCatalog") \
.config("spark.sql.catalog.hadoop_catalog.type", "hadoop") \
.config("spark.sql.catalog.hadoop_catalog.warehouse", "/content/warehouse") \
.config("spark.sql.default.catalog", "hadoop_catalog") \
.getOrCreate()

# Diretório do Warehouse
!mkdir -p /content/warehouse

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,982 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,962 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,310 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,847 kB]
Get:14 http://archive.ubu

# Incorporando Dados Existentes

In [3]:
!unzip vendas_iceberg.zip -d /content/
!mkdir -p /content/warehouse/default/vendas_iceberg
!cp -r /content/vendas_iceberg/* /content/warehouse/default/vendas_iceberg/

Archive:  vendas_iceberg.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of vendas_iceberg.zip or
        vendas_iceberg.zip.zip, and cannot find vendas_iceberg.zip.ZIP, period.
cp: cannot stat '/content/vendas_iceberg/*': No such file or directory


In [4]:
# Criar uma tabela Iceberg diretamente a partir do Parquet existente
spark.sql("DROP TABLE IF EXISTS hadoop_catalog.default.vendas_iceberg")

spark.sql(f"""
    CREATE TABLE hadoop_catalog.default.vendas_iceberg (
        id INT,
        produto STRING,
        quantidade INT,
        preco DOUBLE,
        data_venda DATE
    )
    USING iceberg
""")

DataFrame[]

In [5]:
# le parquet
df_parquet = spark.read.parquet("/content/warehouse/default/vendas_iceberg/")

# atualiza na tabela
df_parquet.writeTo("hadoop_catalog.default.vendas_iceberg").append()

In [6]:
#visualizamos os dados
spark.sql("SELECT * FROM hadoop_catalog.default.vendas_iceberg order by id").show()

+---+---------+----------+-----+----------+
| id|  produto|quantidade|preco|data_venda|
+---+---------+----------+-----+----------+
|  1|Produto A|        10| 15.5|2023-11-01|
|  2|Produto B|         5| 22.0|2023-11-02|
|  3|Produto C|         8| 30.0|2023-11-03|
|  4|Produto D|        12| 25.0|2023-11-04|
|  5|Produto E|         7| 18.5|2023-11-05|
+---+---------+----------+-----+----------+



In [7]:
# mostra snapshots
snapshots_df = spark.sql("SELECT * FROM hadoop_catalog.default.vendas_iceberg.snapshots")
snapshots_df.select("snapshot_id", "committed_at", "operation").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|2234980557864308997|2026-04-07 20:48:18.084|append   |
+-------------------+-----------------------+---------+

